In [ ]:
"""
Filter the expanded Karaman long table by:
  1. Kinase filter: keep only rows whose UniProt_ID appears in the
     KLIFS-curated 484-kinase reference file.
  2. Molecule filter: drop every Karaman molecule whose Morgan fingerprint
     (radius=2, nBits=2048) matches any training-set molecule.
     If a molecule is dropped, ALL rows referencing it are removed.

Outputs:
  - filtered long table (CSV)
  - list of overlapping Karaman molecule names (TXT)
  - step-by-step counts printed to stdout
"""

from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdFingerprintGenerator
from rdkit import RDLogger

# Silence RDKit's noisy SMILES parsing warnings; we handle failures explicitly.
RDLogger.DisableLog("rdApp.*")

# ---------------------------------------------------------------------------
# Paths -- adjust these to your environment.
# ---------------------------------------------------------------------------
KARAMAN_LONG_PATH = "../dataset/Karaman_expand_final.csv"
KINASE_484_PATH   = "../feature_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit.csv"
TRAIN_SMILES_PATH = "../dataset/band_0.10_0.50/train_with_smiles_protein_seq.csv"

OUTPUT_LONG_PATH  = "../dataset/Karaman_Kinase_selectivity_filtered_withseq.csv"

# Morgan fingerprint parameters for IDENTITY matching (not modelling features).
MORGAN_RADIUS = 2
MORGAN_NBITS  = 2048

# Morgan fingerprint parameters for the final stored column (model features).
FINAL_FP_RADIUS = 2
FINAL_FP_NBITS  = 4096
FINAL_FP_COL    = "Morgan"


def smiles_to_fp_bitstring(smiles: str):
    """Return Morgan fingerprint as a bitstring, or None if SMILES is invalid."""
    if not isinstance(smiles, str) or smiles.strip() == "":
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius=MORGAN_RADIUS, nBits=MORGAN_NBITS
    )
    return fp.ToBitString()


def main():
    # ---------------------------------------------------------------------
    # Load inputs
    # ---------------------------------------------------------------------
    karaman = pd.read_csv(KARAMAN_LONG_PATH)
    kinase_ref = pd.read_csv(KINASE_484_PATH, usecols=["UniProt_ID"])
    train = pd.read_csv(TRAIN_SMILES_PATH, usecols=["SMILES"])

    print("=" * 70)
    print("INITIAL STATE")
    print("=" * 70)
    print(f"Karaman long table          : {len(karaman):>7} rows")
    print(f"  unique molecules          : {karaman['molecule_name'].nunique():>7}")
    print(f"  unique UniProt IDs        : {karaman['UniProt_ID'].nunique():>7}")
    print(f"  active / inactive         : "
          f"{(karaman['label']==1).sum()} / {(karaman['label']==0).sum()}")
    print(f"484-kinase reference rows   : {len(kinase_ref):>7}")
    print(f"  unique UniProt IDs        : {kinase_ref['UniProt_ID'].nunique():>7}")
    print(f"Training-set SMILES rows    : {len(train):>7}")

    # ---------------------------------------------------------------------
    # Step 1: filter kinases by UniProt_ID membership in 484 reference
    # ---------------------------------------------------------------------
    valid_uniprots = set(kinase_ref["UniProt_ID"].dropna().unique())
    karaman_step1 = karaman[karaman["UniProt_ID"].isin(valid_uniprots)].copy()

    print()
    print("=" * 70)
    print("STEP 1: KINASE FILTER (UniProt_ID in 484 reference)")
    print("=" * 70)
    print(f"Rows                        : {len(karaman):>7} -> {len(karaman_step1):>7}")
    print(f"Unique molecules            : {karaman['molecule_name'].nunique():>7} "
          f"-> {karaman_step1['molecule_name'].nunique():>7}")
    print(f"Unique UniProt IDs          : {karaman['UniProt_ID'].nunique():>7} "
          f"-> {karaman_step1['UniProt_ID'].nunique():>7}")
    print(f"Active / inactive           : "
          f"{(karaman_step1['label']==1).sum()} / "
          f"{(karaman_step1['label']==0).sum()}")

    # ---------------------------------------------------------------------
    # Step 2: deduplicate SMILES in both sets and compute Morgan FPs
    # ---------------------------------------------------------------------
    # Karaman unique molecules (after step 1)
    karaman_unique = (
        karaman_step1[["molecule_name", "SMILES"]]
        .drop_duplicates(subset=["SMILES"])
        .reset_index(drop=True)
    )
    karaman_unique["fp"] = karaman_unique["SMILES"].apply(smiles_to_fp_bitstring)

    n_karaman_fail = karaman_unique["fp"].isna().sum()
    if n_karaman_fail:
        bad = karaman_unique.loc[karaman_unique["fp"].isna(), "molecule_name"].tolist()
        print(f"\n[warn] {n_karaman_fail} Karaman SMILES failed to parse: {bad}")
        karaman_unique = karaman_unique.dropna(subset=["fp"]).reset_index(drop=True)

    # Training unique SMILES (string-level dedup first to save FP work)
    train_unique_smiles = train["SMILES"].dropna().drop_duplicates().tolist()
    train_fps = set()
    n_train_fail = 0
    for smi in train_unique_smiles:
        fp = smiles_to_fp_bitstring(smi)
        if fp is None:
            n_train_fail += 1
            continue
        train_fps.add(fp)
    if n_train_fail:
        print(f"[warn] {n_train_fail} training SMILES failed to parse (skipped)")

    print()
    print("=" * 70)
    print("STEP 2A: FINGERPRINT COMPUTATION")
    print("=" * 70)
    print(f"Karaman unique SMILES       : {len(karaman_unique)}")
    print(f"Training unique SMILES      : {len(train_unique_smiles)}")
    print(f"Training unique FPs         : {len(train_fps)}")

    # ---------------------------------------------------------------------
    # Step 2B: identify overlapping Karaman molecules
    # ---------------------------------------------------------------------
    karaman_unique["in_train"] = karaman_unique["fp"].isin(train_fps)
    overlap = karaman_unique.loc[karaman_unique["in_train"], "molecule_name"].tolist()

    print()
    print("=" * 70)
    print("STEP 2B: MOLECULE OVERLAP WITH TRAINING SET")
    print("=" * 70)
    print(f"Overlapping Karaman molecules: {len(overlap)}")
    for name in overlap:
        print(f"  - {name}")

    # ---------------------------------------------------------------------
    # Step 2C: drop ALL rows belonging to overlapping molecules
    # ---------------------------------------------------------------------
    karaman_step2 = karaman_step1[~karaman_step1["molecule_name"].isin(overlap)].copy()

    print()
    print("=" * 70)
    print("STEP 2C: MOLECULE FILTER (drop all rows of overlapping molecules)")
    print("=" * 70)
    print(f"Rows                        : {len(karaman_step1):>7} -> {len(karaman_step2):>7}")
    print(f"Unique molecules            : {karaman_step1['molecule_name'].nunique():>7} "
          f"-> {karaman_step2['molecule_name'].nunique():>7}")
    print(f"Unique UniProt IDs          : {karaman_step1['UniProt_ID'].nunique():>7} "
          f"-> {karaman_step2['UniProt_ID'].nunique():>7}")
    print(f"Active / inactive           : "
          f"{(karaman_step2['label']==1).sum()} / "
          f"{(karaman_step2['label']==0).sum()}")

    # ---------------------------------------------------------------------
    # Step 3: compute Morgan 4096-bit fingerprints on the FILTERED long table.
    # Compute once per unique SMILES, then map back to every row.
    # ---------------------------------------------------------------------
    final_gen = rdFingerprintGenerator.GetMorganGenerator(
        radius=FINAL_FP_RADIUS, fpSize=FINAL_FP_NBITS
    )

    def smiles_to_final_fp_bitstring(smiles: str):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return final_gen.GetFingerprint(mol).ToBitString()

    unique_smiles = karaman_step2["SMILES"].drop_duplicates().tolist()
    fp_map = {smi: smiles_to_final_fp_bitstring(smi) for smi in unique_smiles}

    n_final_fail = sum(1 for v in fp_map.values() if v is None)
    if n_final_fail:
        bad_smiles = [s for s, v in fp_map.items() if v is None]
        print(f"\n[warn] {n_final_fail} SMILES failed final FP generation: "
              f"{bad_smiles}")

    karaman_step2[FINAL_FP_COL] = karaman_step2["SMILES"].map(fp_map)

    print()
    print("=" * 70)
    print(f"STEP 3: MORGAN FP (radius={FINAL_FP_RADIUS}, nBits={FINAL_FP_NBITS})")
    print("=" * 70)
    print(f"Unique SMILES fingerprinted : {len(unique_smiles)}")
    print(f"FP string length            : {len(next(v for v in fp_map.values() if v))}")
    print(f"Added column                : '{FINAL_FP_COL}'")

    # ---------------------------------------------------------------------
    # Save
    # ---------------------------------------------------------------------
    karaman_step2.to_csv(OUTPUT_LONG_PATH, index=False)
    print()
    print("=" * 70)
    print("OUTPUTS")
    print("=" * 70)
    print(f"Filtered long table : {OUTPUT_LONG_PATH}")


if __name__ == "__main__":
    main()